## Tokenization

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import urllib.request


from pathlib import Path

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
# URL for the raw tiny shakespeare dataset
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

path = Path('tiny-shakespeare.txt')

# Download the file if it doesn't already exist
if not path.exists():
    print("Downloading tiny-shakespeare.txt...")
    urllib.request.urlretrieve(url, path)
    print("Download complete!")

# Now this will work smoothly
text = path.read_text(encoding='utf-8')
print(f"Success! Loaded {len(text):,} characters.")

Success! Loaded 1,115,394 characters.


In [9]:
print(text[:300])  # Print the first 1000 characters of the text

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us


## Building a character level tokenizer

In [10]:
class CharTokenizer:
    def __init__(self, vocabulary):
        self.chars = sorted(list(set(vocabulary)))
        self.vocab_size = len(self.chars)
        self.token_id_for_char = {char: token_id for token_id, char in enumerate(self.chars)}
        self.char_for_token_id = {token_id: char for token_id, char in enumerate(self.chars)}


    @staticmethod
    def train_from_text(text):
        return CharTokenizer(text)

    def encode(self, text):
        token_ids = []
        for char in text:
            token_ids.append(self.token_id_for_char[char])
        return torch.tensor(token_ids, dtype=torch.long)

    def decode(self, token_ids):
        chars = []
        for token_id in token_ids.tolist():
            chars.append(self.char_for_token_id[token_id])
        return ''.join(chars)

    def vocabulary_size(self):
        return self.vocab_size

In [11]:
tokenizer = CharTokenizer.train_from_text(text)

In [12]:
tokenizer

In [13]:
encoding = tokenizer.encode("Hello World")

print(encoding)

tensor([20, 43, 50, 50, 53,  1, 35, 53, 56, 50, 42])


In [14]:
print(tokenizer.decode(encoding))

Hello World


In [15]:
tokenizer.vocabulary_size()

65

## Implementing a Dataset

In [1]:
from torch.utils.data import Dataset, DataLoader

class CustomDataset(Dataset):
    def __init__(self, data, transform):
        self.data = data
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        if self.transform:
            item = self.transform(item)
        return item

# Create DataLoader
def create_data_loader(data, transform, batch_size):
    dataset = CustomDataset(data, transform)
    return DataLoader(dataset, batch_size=batch_size, shuffle=True)


## TokenIdDataset

In [16]:
class TokenIdsDataset(Dataset):
    def __init__(self, data, block_size):
        # TODO: Save data and block size
        self.data = data
        self.block_size = block_size

    def __len__(self):
        # TODO: If every position can be a start of an item,
        # and all items should be "block_size", compute the size
        # of the dataset
        return len(self.data) - self.block_size

    def __getitem__(self, pos):
        # TODO: Check if the input position is valid
        if pos < 0 or pos >= len(self):
            raise IndexError("Invalid position")
        assert pos < len(self.data) - self.block_size, "Position out of range for block size"

        # TODO: Get an item from position "pos"
        x = self.data[pos:pos + self.block_size]
        # TODO: Get a target item (shifted by one position)
        y = self.data[pos + 1:pos + self.block_size + 1]

        # TODO: Return both
        return x, y

In [17]:
# Step 2 - Tokenize the Text

# TODO: Encode text using the tokenizer
# Create "TokenIdsDataset" with the tokenized text, and block_size=64
tokenized_text = tokenizer.encode(text)
dataset = TokenIdsDataset(tokenized_text, block_size=64)

In [18]:
tokenized_text

tensor([18, 47, 56,  ..., 45,  8,  0])

In [19]:
dataset

In [20]:
x, y = next(iter(dataset))

In [21]:
x

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50])

In [22]:
y

tensor([47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44, 53,
        56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,  1,
        44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1, 57,
        54, 43, 39, 49,  8,  0,  0, 13, 50, 50])

In [23]:
x.shape

torch.Size([64])

In [25]:
tokenizer.decode(x)

'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAl'

In [24]:
y.shape

torch.Size([64])

In [26]:
tokenizer.decode(y)

'irst Citizen:\nBefore we proceed any further, hear me speak.\n\nAll'

In [ ]:
# Step 3 - Retrieve the First Item from the Dataset

# TODO: Get the first item from the dataset
# Decode "x" using tokenizer.decode

In [36]:
# Step 4 - Use a DataLoader

# TODO: Get a single batch from the "dataloader"
# For this call the `iter` function, and pass DataLoader instance to it. This will create an iterator
# Then call the `next` function and pass the iterator to it to get the first training batch

In [27]:
from torch.utils.data import DataLoader, RandomSampler

# RandomSampler allows to read random items from a datasset
sampler = RandomSampler(dataset, replacement=True)
# Dataloader will laod two random samplers using the sampler
dataloader = DataLoader(dataset, batch_size=2, sampler=sampler)

In [28]:
sampler

In [29]:
dataloader

In [30]:
x, y = next(iter(dataloader))

In [37]:
# TODO: Decode input item

In [31]:
x

tensor([[ 1, 44, 47, 50, 50,  1, 51, 63,  1, 40, 43, 50, 50, 63, 11,  1, 53, 52,
         43,  1, 44, 56, 59, 47, 58, 44, 59, 50,  1, 51, 43, 39, 50,  1, 61, 53,
         59, 50, 42,  1, 57, 43, 58,  0, 51, 43,  1, 58, 53,  1,  5, 58,  8,  1,
         14, 59, 58,  1, 58, 46, 43, 63,  1, 57],
        [14, 30, 13, 37, 10,  0, 13, 52, 42,  1, 21,  6,  1, 58, 53,  1, 49, 43,
         43, 54,  1, 39, 50, 50,  1, 58, 46, 47, 57,  8,  0,  0, 20, 17, 26, 30,
         37,  1, 14, 27, 24, 21, 26, 19, 14, 30, 27, 23, 17, 10,  0, 26, 53, 56,
         44, 53, 50, 49,  6,  1, 57, 53,  1, 44]])

In [33]:
tokenizer.decode(x[0])

" fill my belly; one fruitful meal would set\nme to 't. But they s"

In [34]:
tokenizer.decode(x[1])

'BRAY:\nAnd I, to keep all this.\n\nHENRY BOLINGBROKE:\nNorfolk, so f'

In [38]:
# TODO: Decode target item

In [39]:
tokenizer.decode(y[0])

"fill my belly; one fruitful meal would set\nme to 't. But they sa"

In [40]:
tokenizer.decode(y[1])

'RAY:\nAnd I, to keep all this.\n\nHENRY BOLINGBROKE:\nNorfolk, so fa'

In [2]:
images = load_images()

NameError: name 'load_images' is not defined

In [ ]:
images_dataset = CustomDataset(images, transform=resize_image)

dataset_len = len(images_dataset)

first_image = images_dataset[0]

for sample in range(len(images_dataset)):
    print(sample.shape)  # Print the shape of each image
    image = images_dataset[sample]
    break  # Just to check the first sample